# 00. 시작하기 — JupyterLab 과 이 PC 점검

**노트북 쓰는 법**
- 칸(셀)을 누르고 `Shift + Enter` → 그 칸만 실행하고 다음 칸으로 간다.
- 위에서부터 차례로 실행한다. 변수는 칸 사이에 이어진다.
- 이상해지면 메뉴 **Kernel → Restart Kernel** 후 처음부터 다시.
- 저장: `Ctrl + S`. 노트북은 작업폴더(`문서\Edge Lab\notebooks`)에 있다.
  원본으로 되돌리려면 그 노트북 파일을 지우고 `lab.bat` 을 다시 켠다.

**노트북 목록**

| 번호 | 내용 | 핵심 |
|---|---|---|
| 01 | 얼굴 찾기 · CPU/GPU/NPU 비교 | OpenVINO 기본, 장치 비교 |
| 02 | YOLO 직접 추론 | letterbox · 출력 해석 · NMS 를 직접 |
| 03 | 얼굴 분석 | 나이·성별 · 감정 · 머리 방향 (모델 3개 잇기) |
| 04 | 사진 바꾸기 | 배경 지우기 · 화질 4배 · 깊이 |
| 05 | 글자 · QR | easyocr · OpenCV QR |
| 06 | 손 · 얼굴 점 | MediaPipe |
| 07 | 나만의 분류기 | 특징 뽑기 + numpy 로 학습 |
| 08 | 자료 찾아 답하기 (RAG) | 임베딩 · 코사인 유사도 · LLM |
| 09 | 사진 보고 말하기 (VLM) | openvino-genai |
| 10 | 음성 | Whisper 받아쓰기 · Supertonic 읽어 주기 |
| 11 | 보드 시리얼 | pyserial |

## 1. 파이썬과 패키지

In [ ]:
import sys
from importlib import metadata

print("python", sys.version.split()[0])
for pkg in ["openvino", "openvino-genai", "openvino-tokenizers", "opencv-python", "numpy", "onnxruntime",
            "mediapipe", "easyocr", "ultralytics", "sounddevice", "pyserial", "requests", "jupyterlab"]:
    try:
        print(f"  {pkg:22s} {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"  {pkg:22s} (없음)")

## 2. AI 장치와 모델 폴더

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

for d in core.available_devices:
    print(f"{d:6s}", core.get_property(d, "FULL_DEVICE_NAME"))

In [ ]:
# 노트북들이 쓰는 모델 — 없으면 그 노트북만 못 한다
NEED = {
    "01 · 03 얼굴": "face/face-detection-retail-0005.xml",
    "02 YOLO": "object/yolo11m_openvino_model",
    "03 나이·성별": "face/age-gender-recognition-retail-0013.xml",
    "03 감정": "face/emotions-recognition-retail-0003.xml",
    "03 머리 방향": "face/head-pose-estimation-adas-0001.xml",
    "04 배경 지우기": "gan/u2net.onnx",
    "04 화질 4배": "gan/single-image-super-resolution-1032.xml",
    "04 깊이": "gan/depth-v2s.xml",
    "05 글자(easyocr)": "code/easyocr",
    "06 MediaPipe 손": "mediapipe/gesture_recognizer.task",
    "07 특징 뽑기": "backbone/mobilenetv2_feat.xml",
    "08 임베딩": "embed/e5-small-int8",
    "08 · 09 VLM": "vlm",
    "10 받아쓰기": "stt",
    "10 읽어 주기": "tts-int8",
}
for what, rel in NEED.items():
    print(f"{'OK ' if (MODELS / rel).exists() else '없음'}  {what:16s} models/{rel}")

## 3. 웹캠

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 4. 마이크 · 스피커

In [ ]:
try:
    import sounddevice as sd
    print(sd.query_devices())
    print("기본 (입력, 출력):", sd.default.device)
except (ImportError, OSError) as ex:           # 우분투는 PortAudio 가 따로 필요하다
    print("소리 장치를 못 읽었어요:", ex)
    print("우분투: sudo apt install libportaudio2  /  윈도우: pip install -r requirements.lock.txt")

## 주의 — 메모리
edge-lab(`run.bat`)이 켜져 있으면 모델이 이미 메모리에 올라가 있다. 노트북에서 같은 모델을 또 올리면 **두 벌**이 된다
(특히 09 VLM 은 3.5GB). 큰 모델 노트북(08 · 09 · 10)은 edge-lab 을 끄고 한다.